In [ ]:
"""
to be covered in this project:
- What RAG is and the problems it solves: knowledge cutoffs, hallucinations,
  and domain-specific grounding

- How a RAG pipeline is structured: the four stages
  (query, retrieve, augment, generate) and how data flows between them

- Building a working pipeline: connecting ChromaDB and the OpenAI
  Chat Completions API into a functional system

- Prompt design for RAG: structuring prompts so the model uses provided
  documentation and handles unanswerable questions gracefully

- Source attribution: getting the model to cite its sources
"""

In [4]:

import os
import json

import cohere
import chromadb
# from openai import OpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
co = cohere.Client(api_key=os.getenv("COHERE_API_KEY"))
# oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
client = chromadb.PersistentClient(path="Introduction_to_RAG/chroma_scoped")
collection = client.get_collection(name="git_docs_scoped")

print(f"ChromaDB collection: {collection.name}")
print(f"Document count: {collection.count()}")
print("Setup verified.")

/Users/sbamwite/Desktop/rags/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


ChromaDB collection: git_docs_scoped
Document count: 334
Setup verified.


In [5]:

SYSTEM_PROMPT = """You are GitQuest, a Git support agent that helps \
developers use Git correctly and confidently.

Answer the user's question using ONLY the documentation provided below. \
Do not use knowledge from your training data.

Guidelines:
- Provide the exact command syntax as shown in the documentation
- Briefly explain what the command does and why it works
- If there are important options or variations shown in the docs, mention them
- If the provided documentation does not contain enough information to \
answer the question, say so explicitly rather than guessing or drawing \
on outside knowledge

End your answer with a SOURCES section listing only the chunk_ids you \
drew from, in this exact format:

SOURCES:
- chunk_id: <id> | <title>

Documentation:
{context}"""


In [ ]:
corpus = {}
with open("data/git_kb_corpus_scoped/corpus.jsonl", "r") as f:
    for line in f:
        chunk = json.loads(line)
        corpus[chunk["chunk_id"]] = chunk


def retrieve(query, n_results=5):
    response = co.embed(
        texts=[query],
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"]
    )
    query_embedding = response.embeddings.float[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    chunks = []
    for chunk_id, metadata, distance in zip(
        results["ids"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunk = corpus[chunk_id]
        chunks.append({
            "chunk_id": chunk_id,
            "text": chunk["text"],
            "title": chunk["title"],
            "source_type": metadata["source_type"],
            "command": metadata["command"],
            "distance": distance
        })
    return chunks


def build_context(chunks):
    context_parts = []
    for i, chunk in enumerate(chunks, start=1):
        context_parts.append(
            f"[SOURCE {i}]\n"
            f"chunk_id: {chunk['chunk_id']}\n"
            f"title: {chunk['title']}\n"
            f"source_type: {chunk['source_type']}\n"
            f"command: {chunk['command']}\n\n"
            f"{chunk['text']}"
        )
    return "\n\n---\n\n".join(context_parts)


SYSTEM_PROMPT = """You are GitQuest, a Git support agent that helps \
developers use Git correctly and confidently.

Answer the user's question using ONLY the documentation provided below. \
Do not use knowledge from your training data.

Guidelines:
- Provide the exact command syntax as shown in the documentation
- Briefly explain what the command does and why it works
- If there are important options or variations shown in the docs, mention them
- If the provided documentation does not contain enough information to \
answer the question, say so explicitly rather than guessing or drawing \
on outside knowledge

End your answer with a SOURCES section listing only the chunk_ids you \
drew from, in this exact format:

SOURCES:
- chunk_id: <id> | <title>

Documentation:
{context}"""


def parse_citations(raw_answer, retrieved_chunks):
    valid_ids = {c["chunk_id"] for c in retrieved_chunks}
    cited = []
    if "SOURCES:" in raw_answer:
        sources_section = raw_answer.split("SOURCES:")[1]
        for line in sources_section.strip().split("\n"):
            if "chunk_id:" in line:
                cited_id = line.split("chunk_id:")[1].split("|")[0].strip()
                if cited_id in valid_ids:
                    chunk = corpus[cited_id]
                    cited.append({
                        "chunk_id": cited_id,
                        "title": chunk["title"],
                        "command": chunk["command"],
                        "source_type": chunk["source_type"]
                    })
    return cited


def ask_gitquest(query, n_results=5):
    chunks = retrieve(query, n_results=n_results)
    context = build_context(chunks)
    response = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT.format(context=context)
            },
            {"role": "user", "content": query}
        ]
    )
    raw_answer = response.choices[0].message.content
    citations = parse_citations(raw_answer, chunks)
    answer_text = raw_answer.split("SOURCES:")[0].strip()

    return {
        "query": query,
        "answer": answer_text,
        "citations": citations,
        "retrieved_chunks": chunks
    }

In [ ]:
query = "How do I discard all the changes I made to a file and restore it to the last commit?"

def retrieve(query, n_results=5):
    response = co.embed(
        texts=[query],
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"]
    )
    query_embedding = response.embeddings.float[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    chunks = []
    for chunk_id, metadata, distance in zip(
        results["ids"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunk = corpus[chunk_id]
        chunks.append({
            "chunk_id": chunk_id,
            "text": chunk["text"],
            "title": chunk["title"],
            "source_type": metadata["source_type"],
            "command": metadata["command"],
            "distance": distance
        })
    return chunks

In [ ]:
# testing

chunks = retrieve("How do I discard all the changes I made to a file and restore it to the last commit?")

for chunk in chunks:
    print(f"[{chunk['distance']:.4f}] {chunk['source_type']} | {chunk['title']}")
    print(f"  chunk_id: {chunk['chunk_id']}")
    print(f"  text preview: {chunk['text'][:100]}...")
    print()

In [ ]:
queries = [
    "How do I discard all the changes I made to a file and restore it to the last commit?",
    "What is the difference between git switch and git checkout for changing branches?",
    "What does git stash do and when should I use it?"
]

for query in queries:
    result = ask_gitquest(query)
    print(f"QUERY: {result['query']}\n")
    print(f"ANSWER:\n{result['answer']}\n")
    print("Retrieved chunks:")
    for chunk in result["retrieved_chunks"]:
        print(f"  [{chunk['distance']:.4f}] {chunk['source_type']} | {chunk['title']}")
    print("=" * 60 + "\n")

In [ ]:
result = ask_gitquest("What is the difference between git switch and git checkout for changing branches?")
print(f"QUERY: {result['query']}\n")
print(f"ANSWER:\n{result['answer']}\n")
print("CITATIONS:")
for c in result["citations"]:
    print(f"  [{c['source_type']}] {c['chunk_id']} | {c['title']}")
print(f"\nRetrieved {len(result['retrieved_chunks'])} chunks, cited {len(result['citations'])}")

In [ ]:
queries = [
    "How do I discard all the changes I made to a file and restore it to the last commit?",
    "What is the difference between git switch and git checkout for changing branches?",
    "How do I create a new branch and switch to it at the same time?",
    "What happens to my uncommitted changes when I switch branches?",
    "What does git stash do and when should I use it?",
    "How do I configure Git to use a custom SSH key for a specific host?"
]
"""
The six queries cover a range of scenarios:
    1. single command with a clear modern equivalent

    2. comparison between two related commands

    3. straightforward how-to with a specific syntax answer

    4. question about Git's behavior rather than a specific command

    5. question about a utility command with multiple subcommands

    6. question that falls outside the corpus entirely

review:
- Is the answer grounded in what was actually retrieved?

- Do the citations match the chunks that came back?

- Does GitQuest handle the “unanswerable query”
  (the one about a custom SSH key) correctly?
"""

for query in queries:
    result = ask_gitquest(query)
    print(f"QUERY: {result['query']}\n")
    print(f"ANSWER:\n{result['answer']}\n")
    print("CITATIONS:")
    for c in result["citations"]:
        print(f"  [{c['source_type']}] {c['chunk_id']} | {c['title']}")
    print(f"\nRetrieved {len(result['retrieved_chunks'])} chunks, "
          f"cited {len(result['citations'])}")
    print("=" * 60 + "\n")